# Week 95, Day 6: Review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w095d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('slugs.py', 'w', encoding='utf-8') as f:
    f.write('import re\n\n\ndef slugify(title, max_length=40):\n    """Turn a title into a lower-case slug of letters, digits and single hyphens."""\n    slug = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")\n    if len(slug) > max_length:\n        slug = slug[:max_length].rstrip("-")\n    return slug or "untitled"\n')

**Predict before you run:** what will this print?



In [ ]:
from slugs import slugify
print(slugify("Mill to Ridge: a long walk over the hill and back", 21))
print(slugify("???"), slugify("  Café & Lake  "))

**Predict before you run:** what will this print?



In [ ]:
lo, hi, calls = 0, 15, 0
while hi - lo > 1:
    mid = (lo + hi) // 2
    calls += 1
    lo, hi = (lo, mid) if mid >= 9 else (mid, hi)
print(hi, calls)

**Predict before you run:** what will this print?



In [ ]:
words = ["a", "b", "c", "d"]
print(words[:1] + words[2:], words[:2] + words[3:])

**Predict before you run:** what will this print?



In [ ]:
import re
print(re.sub(r"[^a-z0-9]+", "-", "Mill -- to  Ridge".lower()))

**Predict before you run:** what will this print?



In [ ]:
print("@@ -14,7 +14,8 @@".split()[2].lstrip("+").split(","))

### Exercise: Hunk headers

Write `hunk_header(old_start, old_count, new_start, new_count)`: return `@@ -OLD +NEW @@`, where each side is `START,COUNT`, or just `START` when the count is 1.

In [ ]:
def hunk_header(old_start, old_count, new_start, new_count):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_headers():\n    \"git's format\"\n    assert hunk_header(14, 7, 14, 8) == \"@@ -14,7 +14,8 @@\"\n    assert hunk_header(1, 1, 1, 1) == \"@@ -1 +1 @@\"\n    assert hunk_header(0, 0, 1, 2) == \"@@ -0,0 +1,2 @@\"")

### Exercise: NEWS by section

Write `news_by_section(paths)`: from changed file paths, count the NEWS entries (paths under `Misc/NEWS.d/next/`) in each section (the folder after `next/`). Return a dict with sections in alphabetical order.

In [ ]:
def news_by_section(paths):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sections():\n    \"a release's NEWS\"\n    paths = [\"Misc/NEWS.d/next/Library/2026-11-20-14-03-12.gh-issue-77.AbCdEf.rst\", \"Lib/wraptext.py\",\n             \"Misc/NEWS.d/next/Documentation/2026-11-21-09-00-00.gh-issue-80.QwErTy.rst\",\n             \"Misc/NEWS.d/next/Library/2026-11-22-10-00-00.gh-issue-81.ZxCvBn.rst\"]\n    assert news_by_section(paths) == {\"Documentation\": 1, \"Library\": 2}\n    assert list(news_by_section(paths)) == [\"Documentation\", \"Library\"] and news_by_section([]) == {}")

### Exercise: Tests that kill every mutant

Write a `unittest` suite for `slugify` in `slugs.py` that passes on it and catches all six broken versions: the original bug (no strip after cutting), no lower-casing, hyphen runs kept, ends not stripped, cut one character too short, and no `"untitled"` fallback. Include a regression test that mentions the bug in a comment, and use `subTest` for at least one group of cases.

In [ ]:
import unittest
from slugs import slugify

class TestSlugify(unittest.TestCase):
    def test_simple(self):
        self.assertEqual(slugify("walk"), "walk")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"slugs.py\").read()\nBROKEN = {\n    \"the original bug\": GOOD.replace('slug = slug[:max_length].rstrip(\"-\")', \"slug = slug[:max_length]\"),\n    \"no lower-casing\": GOOD.replace(\"title.lower()\", \"title\"),\n    \"hyphen runs kept\": GOOD.replace('r\"[^a-z0-9]+\"', 'r\"[^a-z0-9]\"'),\n    \"ends not stripped\": GOOD.replace('title.lower()).strip(\"-\")', \"title.lower())\"),\n    \"cut too short\": GOOD.replace(\"slug[:max_length]\", \"slug[:max_length - 1]\"),\n    \"no fallback\": GOOD.replace('return slug or \"untitled\"', \"return slug\"),\n}\n\ndef test_suite():\n    \"all six caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    assert \"subTest\" in source, \"use subTest for a group of cases\"\n    assert \"#\" in source, \"say in a comment what the regression test is for\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"slugs\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"slugs.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

> **What the week was really about.** The fix this week was one character, both times. Everything else (the reproduction, the bisection, the tests, the clean diff, the replies) was about making that character trustworthy to people who've never met you. That's what contributing is: most of the work is making your change easy to accept. Which part took you longest on your own issue?

### Exercise: Bonus: resolving conflict blocks

Write `resolve(text, choose)`: `text` may contain conflict blocks, each a line `<<<<<<< ...`, our lines, a line `=======`, their lines, and a line `>>>>>>> ...`. Replace each block with the lines returned by `choose(ours, theirs)` (two lists of lines). Return `(resolved_text, blocks)`, keeping a final newline if `text` had one. Raise `ValueError("unfinished conflict")` if a block isn't closed, and `ValueError("stray conflict marker")` for a `=======` or `>>>>>>>` line outside a block.

In [ ]:
def resolve(text, choose):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("TEXT = \"\"\"def shorten(text, width, placeholder=\" [...]\"):\n<<<<<<< ours\n    if length + extra + len(placeholder) > width:\n=======\n    if length + extra + len(placeholder) >= max_width:\n>>>>>>> main\n        break\n<<<<<<< ours\n    return \" \".join(kept) + placeholder\n=======\n    return \" \".join(kept) + placeholder  # done\n>>>>>>> main\n\"\"\"\n\ndef test_resolve():\n    \"keep both intentions\"\n    keep_ours = lambda ours, theirs: ours\n    resolved, blocks = resolve(TEXT, keep_ours)\n    assert blocks == 2 and \"<<<<<<<\" not in resolved and resolved.endswith(\"placeholder\\n\")\n    by_hand = lambda ours, theirs: [o.replace(\"width\", \"max_width\") if \">\" in o else t for o, t in zip(ours, theirs)]\n    resolved, _ = resolve(TEXT, by_hand)\n    assert resolved.splitlines()[1] == \"    if length + extra + len(placeholder) > max_width:\"\n    assert resolve(\"no conflicts\", keep_ours) == (\"no conflicts\", 0)\n\ndef test_malformed():\n    \"markers that don't match\"\n    for bad, message in [(\"<<<<<<< a\\nx\\n=======\\ny\\n\", \"unfinished conflict\"), (\"x\\n=======\\ny\\n\", \"stray conflict marker\"),\n                         (\"x\\n>>>>>>> main\\n\", \"stray conflict marker\")]:\n        try:\n            resolve(bad, lambda o, t: o)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(message)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## A second bug

The report: "`slugify('Mill to Ridge: a long walk over the hill and back', 21)` gives `mill-to-ridge-a-long-`, with a hyphen at the end." Reproduced on `main`; minimal case: any title whose slug, cut at `max_length`, ends at a hyphen. Cause: the slug was cut after stripping hyphens from its ends, so the cut could make a new one. The fix (in `slugs.py`) strips again after cutting: `.rstrip("-")`. What's left is the part reviewers check first: tests that fail without the fix, and that would catch the other mistakes someone might make in this function.

## Bonus: resolving conflicts

## Weekly quiz

## Week 95 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can confirm a reported bug, minimise it, bisect it and write a clear reproduction. ([Day 1](w095d1.html))
- I can explain a bug's cause, make the smallest fix, and check that nothing else changed. ([Day 2](w095d2.html))
- I can write unittest regression and boundary tests, and measure them with mutants. ([Day 3](w095d3.html))
- I can write a commit message and description, read my own diff, and check a PR before opening it. ([Day 4](w095d4.html))
- I can sort review comments, apply suggestions, track threads, reply well and resolve conflicts. ([Day 5](w095d5.html))
- I can test a second fix so that no plausible mistake survives, and resolve conflict blocks. (Today)

## Your turn

1. **Finish your pull request**: every comment answered, every thread waiting on the reviewer or resolved.
2. **Pick a second issue** for next week's contribution sprint, with last week's scoring, so you're ready to start.
3. **Note what slowed you down** this week; next week has three sprint days to do it again, faster.

Tomorrow is a **rest day**. Next week is **Open source III**: three contribution sprint days, reviewing other people's pull requests, and writing up what you contributed.